In [ ]:
import glob
import tqdm

import pandas as pd
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib import rcParams

from utils import *

EXPERIMENT_RESULTS_PATH = '../IS26-experiments/'
TYPE_OF_METRICS = 'bootstrapped_metrics-n_1000-stratified' # 'metrics'

rcParams.update({
    "font.size":        16,        
    "axes.titlesize":   16,
    "axes.labelsize":   16,
    "xtick.labelsize":  16,
    "ytick.labelsize":  16,
    "legend.fontsize":  16,
    "figure.titlesize": 16,
})


In [ ]:
def get_amount_samples(experiment_path):
    splits_path = None
    for path in Path(experiment_path).parents:
        if len(list(path.glob('splits-tvt-*.pkl'))) > 0:
            splits_path = list(path.glob('splits-tvt-*.pkl'))[0]
            break
    if not splits_path:
        return 0
    splits = pd.read_pickle(splits_path)
    acum_test = []
    for r in splits[0]:
        acum_test.extend(r[1])
    acum_train = []
    for r in splits[0]:
        acum_train.extend(r[0])
    assert len(set(acum_train)) == len(set(acum_test)), "Train and test samples are not disjoint"
    return len(set(acum_test))    

In [ ]:
results = []
config_columns = []
for experiment_path in tqdm.tqdm(glob.glob(f'{EXPERIMENT_RESULTS_PATH}/**/{TYPE_OF_METRICS}.pkl', recursive=True)):
    amount_samples = get_amount_samples(experiment_path)
    if amount_samples == 0:
        print(f"Could not find splits file for experiment: {experiment_path}. Skipping...")
        continue
    df = pd.read_pickle(experiment_path)
    experiment_config = read_config_from_path(experiment_path, EXPERIMENT_RESULTS_PATH)
    for column, value in experiment_config.items():
        df[column] = value
    df['amount_samples'] = amount_samples
    results.append(df)
    config_columns.extend(experiment_config.keys())

if len(results) == 0:
    raise Warning(f"No results found in the specified path: {EXPERIMENT_RESULTS_PATH}")

results = pd.concat(results, ignore_index=True)
config_columns = list(set(config_columns))
for c in set(config_columns):
    if c:
        results[c] = results[c].fillna('')


columns_to_ignore = ['dataset', 'subset', 'feature', 'resampled']
cols = list(set(config_columns) - set(columns_to_ignore))
results['config_label']  =  results[cols].apply(lambda row: '\n'.join([str(c) for c in row if c]), axis=1)
for c in cols:
    root_common_tokens = get_common_tokens_with_root(list(results[c].unique()))
    results[f'short_{c}'] = results[c].map(lambda name: compute_short_name_with_root(name, root_common_tokens))

results['config_xlabel'] = results.apply(lambda row: '\n'.join([str(row[f'short_{c}']) for c in cols if row[c]]), axis=1)
results['config_xlabel'] = results['config_xlabel'] + results['aligner'].apply(lambda x: '\nNon-Speech' if 'non_speech' in x else '')
results['feature_label'] = results['feature'].str.split('-').str[0].str.upper()

In [ ]:
results.groupby(['dataset', 'subset', 'feature_label', 'config_xlabel']).size().reset_index().sort_values('dataset').head()

In [ ]:
print('Define colors for:', results.subset.unique())
colors = {
    'enhanced-original': '#4682B4',          
    'original': "#7BCD5A",       
}

In [ ]:
bar_width = 0.2
metric = 'auc'
ticks = np.arange(0, 1, 0.2)

for dataset_name, dataset_df in results.groupby('dataset'):
    feature_order = dataset_df['feature_label'].unique().tolist()
    subset_order = dataset_df['subset'].unique().tolist()

    fig, axes = plt.subplots(nrows=len(feature_order), figsize=(16, 4 * len(feature_order)), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, feature_name in zip(axes, feature_order):
        feature_df = dataset_df[dataset_df["feature_label"] == feature_name]

        config_values = feature_df['config_xlabel'].dropna()
        unique_configs = config_values.unique().tolist()
        config_position = {config: i for i, config in enumerate(unique_configs)}


        for config_name, config_df in feature_df.groupby('config_xlabel'):
            i = config_position[config_name]
            for idx, subset in enumerate(subset_order):
                subset_df = config_df[config_df["subset"] == subset]
                if subset_df.empty:
                    continue

                offset = (idx - (len(subset_order) - 1) / 2) * bar_width
                x = i + offset
                mean_value = subset_df[metric].mean()

                ax.bar(x, mean_value, width=bar_width, color=colors[subset])

                err_min_value = subset_df[metric].quantile(0.05)
                err_max_value = subset_df[metric].quantile(0.95)
                ax.errorbar(
                    x, mean_value,
                    yerr=[[mean_value - err_min_value], [err_max_value - mean_value]],
                    fmt="none", ecolor="black", capsize=3, elinewidth=1
                )

                random_value = subset_df[f'random_{metric}'].mean() if f'random_{metric}' in subset_df.columns else 0.5
                ax.hlines(
                    y=random_value,
                    xmin=x - bar_width / 2,
                    xmax=x + bar_width / 2,
                    linestyles="--",
                    linewidth=1,
                    colors="indianred",
                    alpha=1,
                    zorder=5
                )

                centered_dist = subset_df[metric] - random_value
                if 0 < np.percentile(centered_dist, 0.1) or 0 > np.percentile(centered_dist, 99.9):
                    ax.text(x, 0, '***', ha='center', va='bottom', color='black', fontsize=14, fontweight='bold')
                elif 0 < np.percentile(centered_dist, 1) or 0 > np.percentile(centered_dist, 99):
                    ax.text(x, 0, '**', ha='center', va='bottom', color='black', fontsize=14, fontweight='bold')
                elif 0 < np.percentile(centered_dist, 5) or 0 > np.percentile(centered_dist, 95):
                    ax.text(x, 0, '*', ha='center', va='bottom', color='black', fontsize=14, fontweight='bold')

        ax.set_xticks(range(len(unique_configs)))
        ax.set_xticklabels(unique_configs, rotation=0, ha='center')
        ax.set_ylim(0, 0.95)
        ax.set_yticks(ticks)
        ax.set_xlim(-0.5, len(unique_configs) - 0.5)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_ylabel(f"{feature_name.upper()} - {metric.upper()}")

    for ax in axes[:-1]:
        ax.tick_params(axis="x", labelbottom=False)
    
    
    subset_handles = [mpatches.Patch(color=colors[subset], label=subset) for subset in subset_order]
    random_handle = mlines.Line2D([], [], color="indianred", linestyle="--", linewidth=1, label="Random baseline")

    fig.legend(handles=subset_handles + [random_handle], loc="upper left", bbox_to_anchor=(0.98, 0.98), 
               frameon=False, ncol=1)


    fig.suptitle(f"{dataset_name.upper()} Dataset", fontweight="bold", fontsize=16)
    fig.tight_layout()
    fig.savefig(f"{EXPERIMENT_RESULTS_PATH}/{dataset_name}-{metric}.pdf", bbox_inches="tight")
